# Preparation

In [1]:
!pip install spacy sacrebleu rouge-score --quiet
!python -m spacy download en_core_web_sm
# NOTE: Ollama needs to be installed and running:
# https://ollama.com/download
# And a model pulled, e.g.:
# ollama pull mistral



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


# Data Generation Test

In [2]:
import subprocess
import spacy
from rouge_score import rouge_scorer
import sacrebleu

# Load English tokenizer from spaCy
nlp = spacy.load("en_core_web_sm")

# 🔁 Step 1: Function to query Ollama locally
def query_ollama(prompt, model='mistral'):
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode(),
        stdout=subprocess.PIPE
    )
    return result.stdout.decode().strip()

# 📝 Step 2: Generate a paraphrase using Ollama
def generate_paraphrase(sentence):
    prompt = f"Paraphrase the following sentence syntactically, but preserve the meaning:\n\"{sentence}\""
    return query_ollama(prompt)

# 🧪 Step 3: Tokenizer using spaCy
def tokenize(text):
    return [token.text for token in nlp(text)]

# 🎯 Step 4: Evaluation using BLEU and ROUGE
def evaluate(reference, generated):
    # Tokenization for BLEU
    reference_tokens = tokenize(reference)
    generated_tokens = tokenize(generated)

    # Convert tokens back to string for sacreBLEU
    bleu_score = sacrebleu.sentence_bleu(" ".join(generated_tokens), [" ".join(reference_tokens)]).score

    # ROUGE Score (uses untokenized strings)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated)

    return {
        "BLEU": bleu_score,
        "ROUGE-1": rouge['rouge1'].fmeasure,
        "ROUGE-L": rouge['rougeL'].fmeasure
    }

# 🧪 Step 5: Example usage
if __name__ == "__main__":
    sentence = "The quick brown fox jumps over the lazy dog."
    print("\n🔹 Original:", sentence)

    paraphrased = generate_paraphrase(sentence)
    print("🔹 Paraphrased:", paraphrased)

    scores = evaluate(sentence, paraphrased)
    print("\n📊 Evaluation Scores:")
    for metric, score in scores.items():
        print(f"{metric}: {score:.4f}")



🔹 Original: The quick brown fox jumps over the lazy dog.


⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠼ ⠴ ⠧ ⠧ ⠏ ⠋ ⠋ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠦ ⠧ ⠇ ⠋ ⠋ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠧ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠼ ⠴ ⠦ ⠧ ⠏ ⠋ ⠋ ⠙ ⠸ ⠼ ⠴ ⠴ ⠦ ⠇ 

🔹 Paraphrased: "The swift tan fox leaps above the sluggish hound."

📊 Evaluation Scores:
BLEU: 4.7892
ROUGE-1: 0.3333
ROUGE-L: 0.3333


# Multiple LLMs Paraphrase

In [3]:
# Cell 2: Imports and setup

import subprocess
import spacy
from rouge_score import rouge_scorer
import sacrebleu
import pandas as pd

# Load spaCy tokenizer
nlp = spacy.load("en_core_web_sm")

# List of LLM models to compare
models = ["mistral", "llama3", "gemma2"]

# Dataset: list of sentences to paraphrase
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is an exciting field of AI.",
    "She sells seashells by the seashore.",
    "Data science involves statistics, programming, and domain knowledge.",
    # Add more sentences as needed
]

In [4]:
# Cell 3: Helper functions

def query_ollama(prompt, model):
    """Run ollama CLI to get a model response."""
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode(),
        stdout=subprocess.PIPE
    )
    return result.stdout.decode().strip()

def generate_paraphrase(sentence, model):
    prompt = f"Paraphrase the following sentence syntactically but preserve the meaning:\n\"{sentence}\""
    return query_ollama(prompt, model)

def tokenize(text):
    return [token.text for token in nlp(text)]

def evaluate(reference, generated):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)

    bleu = sacrebleu.sentence_bleu(" ".join(gen_tokens), [" ".join(ref_tokens)]).score

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated)

    return bleu, rouge['rouge1'].fmeasure, rouge['rougeL'].fmeasure



In [5]:
# Cell 4: Run paraphrasing and evaluation loop

results = []

for sentence in sentences:
    for model in models:
        paraphrase = generate_paraphrase(sentence, model)
        bleu, rouge1, rougeL = evaluate(sentence, paraphrase)
        results.append({
            "sentence": sentence,
            "model": model,
            "paraphrase": paraphrase,
            "BLEU": bleu,
            "ROUGE-1": rouge1,
            "ROUGE-L": rougeL
        })

# Convert to DataFrame for easy analysis
df = pd.DataFrame(results)
df.head()


⠙ ⠙ ⠙ ⠹ ⠼ ⠼ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠸ ⠼ ⠴ ⠦ ⠦ ⠇ ⠏ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠋ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠇ ⠏ ⠋ ⠋ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠼ ⠼ ⠴ ⠦ ⠧ ⠇ ⠋ ⠙ ⠹ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠏ ⠋ ⠙ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠏ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠙ ⠙ ⠙ ⠙ ⠹ ⠙ ⠙ ⠹ ⠙ ⠹ ⠙ ⠋ ⠙ ⠙ ⠹ 

,sentence,model,paraphrase,BLEU,ROUGE-1,ROUGE-L
0,The quick brown fox jumps over the lazy dog.,mistral,"""The swift tan fox leaps above the indolent ho...",4.789232,0.333333,0.333333
1,The quick brown fox jumps over the lazy dog.,llama3,Here is a paraphrased version of the sentence:...,2.007289,0.219512,0.146341
2,The quick brown fox jumps over the lazy dog.,gemma2,Here are a few ways to paraphrase the sentence...,2.797545,0.233333,0.200000
3,Natural language processing is an exciting fie...,mistral,The area of artificial intelligence centered a...,1.226545,0.140845,0.140845
4,Natural language processing is an exciting fie...,llama3,"Here's a paraphrased version:\n\n""The study of...",2.020787,0.202899,0.173913


In [6]:
# Cell 5: Aggregate and compare results

summary = df.groupby("model").agg({
    "BLEU": ["mean", "std"],
    "ROUGE-1": ["mean", "std"],
    "ROUGE-L": ["mean", "std"]
}).round(4)

print("Summary Evaluation Scores per Model:\n")
print(summary)


Summary Evaluation Scores per Model:

           BLEU         ROUGE-1         ROUGE-L        
           mean     std    mean     std    mean     std
model                                                  
gemma2   3.3505  0.6388  0.1903  0.0449  0.1749  0.0353
llama3   1.5712  0.7342  0.1652  0.0561  0.1335  0.0423
mistral  3.4115  1.5317  0.2652  0.0878  0.2252  0.0909


In [7]:
# Cell 6: Display some example paraphrases side-by-side

for model in models:
    print(f"\n=== Examples from model: {model} ===\n")
    sample = df[df.model == model].sample(3, random_state=42)
    for idx, row in sample.iterrows():
        print(f"Original: {row['sentence']}")
        print(f"Paraphrase: {row['paraphrase']}")
        print(f"BLEU: {row['BLEU']:.2f}, ROUGE-1: {row['ROUGE-1']:.2f}, ROUGE-L: {row['ROUGE-L']:.2f}\n")



=== Examples from model: mistral ===

Original: Natural language processing is an exciting field of AI.
Paraphrase: The area of artificial intelligence centered around understanding human la
language is quite intriguing.

OR

The study of artificial intelligence focused on interpreting natural langua
language is captivating.

OR

Artificial Intelligence that deals with comprehending human speech is full 
of interest.

OR

The domain of AI that focuses on natural language understanding is thrillin
thrilling.
BLEU: 1.23, ROUGE-1: 0.14, ROUGE-L: 0.14

Original: Data science involves statistics, programming, and domain knowledge.
Paraphrase: "Statistical analysis, coding skills, and familiarity with specific fields
fields are integral to data science."
BLEU: 3.96, ROUGE-1: 0.32, ROUGE-L: 0.16

Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "The swift tan fox leaps above the indolent hound."
BLEU: 4.79, ROUGE-1: 0.33, ROUGE-L: 0.33


=== Examples from model: llama3 ===


# Multiple LLMs with various temperature settings

In [8]:
!pip install langchain ollama spacy pandas sacrebleu rouge-score --quiet
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.4 MB/s  0:00:09m0:00:0100:01

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [9]:
# Imports
import pandas as pd
import spacy
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from rouge_score import rouge_scorer
import sacrebleu

# Load SpaCy tokenizer
nlp = spacy.load("en_core_web_sm")

# Settings
models = ["mistral", "llama3", "gemma2"]
temperatures = [0.2, 0.5, 0.8]

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is an exciting field of AI.",
    "She sells seashells by the seashore.",
    "Data science involves statistics, programming, and domain knowledge.",
    "A psychologically rich lifestyle, characterized by diverse and stimulating experiences, may foster cognitive benefits such as adaptability and intellectual growth."
]


In [10]:
def create_ollama_model(model_name, temperature):
    return ChatOllama(model=model_name, temperature=temperature)

def generate_paraphrase(model, sentence):
    prompt = (
        "Paraphrase the following sentence. Provide only the result without any explanations. Make syntactic changes but keep the original meaning:\n\n"
        f"\"{sentence}\""
    )
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

def tokenize(text):
    return [token.text for token in nlp(text)]

def evaluate(reference, generated):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)

    bleu = sacrebleu.sentence_bleu(" ".join(gen_tokens), [" ".join(ref_tokens)]).score

    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    scores = scorer.score(reference, generated)

    return bleu, scores["rouge1"].fmeasure, scores["rougeL"].fmeasure


In [11]:
results = []

for model_name in models:
    for temp in temperatures:
        llm = create_ollama_model(model_name, temp)
        for sentence in sentences:
            try:
                paraphrase = generate_paraphrase(llm, sentence)
                bleu, rouge1, rougeL = evaluate(sentence, paraphrase)

                results.append({
                    "model": model_name,
                    "temperature": temp,
                    "sentence": sentence,
                    "paraphrase": paraphrase,
                    "BLEU": bleu,
                    "ROUGE-1": rouge1,
                    "ROUGE-L": rougeL
                })
            except Exception as e:
                print(f"Error for model={model_name}, temp={temp}: {e}")


/var/folders/t1/h8zv1wts1wgf_g3py93d2x440000gn/T/ipykernel_9650/1568216370.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  return ChatOllama(model=model_name, temperature=temperature)


In [12]:
df = pd.DataFrame(results)

summary = df.groupby(["model", "temperature"]).agg({
    "BLEU": ["mean", "std"],
    "ROUGE-1": ["mean", "std"],
    "ROUGE-L": ["mean", "std"]
}).round(4)

summary

BLEU          ROUGE-1         ROUGE-L        
                        mean      std    mean     std    mean     std
model   temperature                                                  
gemma2  0.2          10.0985   6.5921  0.4257  0.1754  0.3527  0.1207
        0.5           8.4476   4.1532  0.3994  0.1144  0.3594  0.1051
        0.8          12.3466  11.2853  0.4562  0.2035  0.3549  0.1606
llama3  0.2          11.4377   5.0205  0.5881  0.2117  0.3741  0.1082
        0.5           9.4719   5.9831  0.4896  0.2594  0.2833  0.0835
        0.8          11.5143   5.5508  0.7415  0.1608  0.4091  0.0888
mistral 0.2           5.8402   2.4831  0.2981  0.1145  0.2332  0.0770
        0.5           4.6038   0.8424  0.2622  0.0708  0.2161  0.0602
        0.8           4.6415   0.6953  0.3149  0.1459  0.2352  0.0985

In [13]:
for model_name in models:
    for temp in temperatures:
        sample = df[(df["model"] == model_name) & (df["temperature"] == temp)].head(2)
        print(f"\n=== {model_name} @ temperature {temp} ===")
        for _, row in sample.iterrows():
            print(f"Original: {row['sentence']}")
            print(f"Paraphrase: {row['paraphrase']}")
            print(f"BLEU: {row['BLEU']:.2f}, ROUGE-1: {row['ROUGE-1']:.2f}, ROUGE-L: {row['ROUGE-L']:.2f}\n")



=== mistral @ temperature 0.2 ===
Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "Swiftly, the reddish canine leaps above the sluggish hound."
BLEU: 3.67, ROUGE-1: 0.22, ROUGE-L: 0.22

Original: Natural language processing is an exciting field of AI.
Paraphrase: Artificial Intelligence's intriguing domain encompasses natural language processing.
BLEU: 9.29, ROUGE-1: 0.33, ROUGE-L: 0.33


=== mistral @ temperature 0.5 ===
Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "Swiftly, the reddish hound leaps above the indolent puppy."
BLEU: 3.67, ROUGE-1: 0.22, ROUGE-L: 0.22

Original: Natural language processing is an exciting field of AI.
Paraphrase: Artificial Intelligence's intriguing domain encompasses natural language handling.
BLEU: 4.99, ROUGE-1: 0.22, ROUGE-L: 0.22


=== mistral @ temperature 0.8 ===
Original: The quick brown fox jumps over the lazy dog.
Paraphrase: The agile red fox leaps above the idle blue hound.
BLEU: 5.30, ROUGE-1: 0.32, R

# Paraphrase LLMs using benchmark dataset

In [14]:
!pip install langchain transformers datasets ollama datasets spacy sacrebleu rouge-score pandas bert-score --quiet
!python -m spacy download en_core_web_sm

# Ollama needs to be installed and running:
# https://ollama.com/download
# And a model pulled on your terminal, e.g.: ollama pull mistral
# It's OK to select small models for the learning purpose


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 962.7 kB/s  0:00:130:00:0100:01

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


I use the following dataset as the benchmark: https://huggingface.co/datasets/impresso-project/amr-true-paraphrases

In [16]:
from datasets import load_dataset
import pandas as pd
import time

from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage

from rouge_score import rouge_scorer
import sacrebleu

from bert_score import score as bert_score

In [17]:
# Load benchmark dataset
dataset = load_dataset("impresso-project/amr-true-paraphrases", split="test[55:65]")  # use a subset for speed

# Extract original and reference paraphrase
df_data = pd.DataFrame(dataset)[["sentence1", "sentence2"]]
df_data = df_data.rename(columns={"sentence1": "original", "sentence2": "reference"}) # the reference is the translated text used as the benchmark

df_data.head()


,original,reference
0,The boy must not go.,It’s obligatory that the boy not go.
1,The boy thinks his team won’t win.,The boy doesn’t think his team will win.
2,It’s not true that the boy thinks his team wil...,The boy doesn’t think his team will win.
3,I don’t have any money.,I have no money.
4,the dress is inappropriate,the dress is not appropriate


In [19]:
# 1. LLMs setting

models = ["mistral", "llama3", "gemma2"]

temperatures = [0.2, 0.5, 0.8]


def create_ollama_model(model_name, temperature):

    return ChatOllama(
        model=model_name,
        temperature=temperature
    )


def generate_paraphrase(model, sentence):

    prompt = (
        "Paraphrase the following sentence.\n"
        "Provide only the paraphrased sentence.\n"
        "Make syntactic changes while preserving meaning.\n\n"
        f"{sentence}"
    )

    try:

        response = model.invoke(
            [HumanMessage(content=prompt)]
        )

        return response.content.strip()

    except Exception as e:

        print(e)

        return ""


# 2. BLEU + ROUGE EVALUATION


rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)


def evaluate(reference, generated):

    # BLEU

    bleu = sacrebleu.sentence_bleu(
        generated,
        [reference]
    ).score

    # ROUGE

    scores = rouge.score(
        reference,
        generated
    )

    rouge1 = scores["rouge1"].fmeasure

    rougeL = scores["rougeL"].fmeasure

    return bleu, rouge1, rougeL



# 3. GENERATE PARAPHRASES

results = []

for i, row in df_data.iterrows():

    original = row["original"]

    reference = row["reference"]

    for model_name in models:

        for temp in temperatures:

            print(
                f"Example {i+1}/{len(df_data)} | "
                f"{model_name} @ {temp}"
            )

            llm = create_ollama_model(
                model_name,
                temp
            )

            generated = generate_paraphrase(
                llm,
                original
            )

            bleu, rouge1, rougeL = evaluate(
                reference,
                generated
            )

            results.append({

                "example_id": i,

                "model": model_name,

                "temperature": temp,

                "original": original,

                "reference": reference,

                "paraphrase": generated,

                "BLEU": bleu,

                "ROUGE-1": rouge1,

                "ROUGE-L": rougeL

            })

            time.sleep(1)


# 4. CONVERT TO DATAFRAME

df_results = pd.DataFrame(results)


# 5. BERTSCORE

print("Computing BERTScore...")

P, R, F1 = bert_score(

    cands=df_results["paraphrase"].tolist(),

    refs=df_results["reference"].tolist(),

    lang="en",

    model_type="roberta-large",

    verbose=True

)

df_results["BERTScore_P"] = P.numpy()

df_results["BERTScore_R"] = R.numpy()

df_results["BERTScore_F1"] = F1.numpy()


Example 1/10 | mistral @ 0.2
Example 1/10 | mistral @ 0.5
Example 1/10 | mistral @ 0.8
Example 1/10 | llama3 @ 0.2
Example 1/10 | llama3 @ 0.5
Example 1/10 | llama3 @ 0.8
Example 1/10 | gemma2 @ 0.2
Example 1/10 | gemma2 @ 0.5
Example 1/10 | gemma2 @ 0.8
Example 2/10 | mistral @ 0.2
Example 2/10 | mistral @ 0.5
Example 2/10 | mistral @ 0.8
Example 2/10 | llama3 @ 0.2
Example 2/10 | llama3 @ 0.5
Example 2/10 | llama3 @ 0.8
Example 2/10 | gemma2 @ 0.2
Example 2/10 | gemma2 @ 0.5
Example 2/10 | gemma2 @ 0.8
Example 3/10 | mistral @ 0.2
Example 3/10 | mistral @ 0.5
Example 3/10 | mistral @ 0.8
Example 3/10 | llama3 @ 0.2
Example 3/10 | llama3 @ 0.5
Example 3/10 | llama3 @ 0.8
Example 3/10 | gemma2 @ 0.2
Example 3/10 | gemma2 @ 0.5
Example 3/10 | gemma2 @ 0.8
Example 4/10 | mistral @ 0.2
Example 4/10 | mistral @ 0.5
Example 4/10 | mistral @ 0.8
Example 4/10 | llama3 @ 0.2
Example 4/10 | llama3 @ 0.5
Example 4/10 | llama3 @ 0.8
Example 4/10 | gemma2 @ 0.2
Example 4/10 | gemma2 @ 0.5
Example 

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 2/2 [00:03<00:00,  1.76s/it]


computing greedy matching.


100%|██████████| 2/2 [00:00<00:00, 146.68it/s]

done in 3.53 seconds, 25.49 sentences/sec


In [20]:
# 6. SUMMARY

summary = (

    df_results

    .groupby(

        ["model", "temperature"]

    )

    .agg({

        "BLEU": ["mean", "std"],

        "ROUGE-1": ["mean", "std"],

        "ROUGE-L": ["mean", "std"],

        "BERTScore_F1": ["mean", "std"]

    })

    .round(4)

)

print(summary)

                        BLEU          ROUGE-1         ROUGE-L          \
                        mean      std    mean     std    mean     std   
model   temperature                                                     
gemma2  0.2          13.4500  14.1729  0.3694  0.2698  0.3694  0.2698   
        0.5          12.5046  14.3718  0.3346  0.2581  0.3172  0.2565   
        0.8          14.3664  14.4517  0.3254  0.3035  0.3111  0.2938   
llama3  0.2          15.8563   9.5925  0.4702  0.2094  0.4326  0.2279   
        0.5          12.7941   7.9135  0.3935  0.1278  0.3700  0.1336   
        0.8           8.8900   5.6445  0.4184  0.1690  0.3823  0.1732   
mistral 0.2           7.6655   3.9264  0.3374  0.2027  0.2644  0.1725   
        0.5           7.5906   3.2514  0.3513  0.1886  0.2683  0.1468   
        0.8           7.0627   4.9354  0.3348  0.1896  0.2618  0.1556   

                    BERTScore_F1          
                            mean     std  
model   temperature                  

In [21]:

# 7. OPTIONAL: RANK MODELS

ranking = (

    df_results

    .groupby(

        ["model", "temperature"]

    )

    ["BERTScore_F1"]

    .mean()

    .sort_values(

        ascending=False

    )

)

print("\nRanking by BERTScore")

print(ranking)


Ranking by BERTScore
model    temperature
llama3   0.2            0.928258
gemma2   0.2            0.927304
llama3   0.5            0.924238
mistral  0.8            0.922935
gemma2   0.8            0.922104
         0.5            0.921455
mistral  0.2            0.920678
llama3   0.8            0.917589
mistral  0.5            0.912267
Name: BERTScore_F1, dtype: float32


In [23]:
for model_name in models:
    for temp in temperatures:
        print(f"\n=== Examples from model: {model_name} @ temp {temp} ===")
        subset = df_results[(df_results.model == model_name) & (df_results.temperature == temp)].sample(3)
        for _, row in subset.iterrows():
            print(f"Original:  {row['original']}")
            print(f"Reference: {row['reference']}")
            print(f"Paraphrase:{row['paraphrase']}")
            print(f"BLEU: {row['BLEU']:.2f}, ROUGE-1: {row['ROUGE-1']:.2f}, ROUGE-L: {row['ROUGE-L']:.2f}, BERTScore_F1: {row['BERTScore_F1']:.2f}\n")


=== Examples from model: mistral @ temp 0.2 ===
Original:  The boy thinks his team won’t win.
Reference: The boy doesn’t think his team will win.
Paraphrase:He believes that his group is unlikely to emerge victorious.
BLEU: 4.46, ROUGE-1: 0.11, ROUGE-L: 0.11, BERTScore_F1: 0.92

Original:  It’s not true that the boy thinks his team will win.
Reference: The boy doesn’t think his team will win.
Paraphrase:The boy doesn't believe his team is going to triumph.
BLEU: 11.21, ROUGE-1: 0.60, ROUGE-L: 0.60, BERTScore_F1: 0.97

Original:  I don’t have any money.
Reference: I have no money.
Paraphrase:I'm currently lacking funds.
BLEU: 10.68, ROUGE-1: 0.22, ROUGE-L: 0.22, BERTScore_F1: 0.93


=== Examples from model: mistral @ temp 0.5 ===
Original:  The boy doesn’t know that the girl came.
Reference: The boy doesn’t know the girl came.
Paraphrase:The girl's arrival isn't recognized by the boy.
BLEU: 6.74, ROUGE-1: 0.56, ROUGE-L: 0.33, BERTScore_F1: 0.92

Original:  the dress is inappropriate
Re

# Evaluation & Interpretation

Automatic quality metrics are divided into string-based metrics and machine learning-based metrics.

String-based metrics generally measure the word or character distance between the target sentence and the reference translation.
Examples: BLEU, ROUGE, METEOR, etc

Machine learning-based metrics use sentence embeddings to calculate the difference between the generated target sentence and the reference translation, or even between the target sentence and the source sentence.
Examples: COMET, YiSi, BERTscore

In this tutorial, a good paraphrase should satisfy two objectives simultaneously:

Paraphrase Quality=Semantic Similarity+Lexical Diversity

| Model     | BLEU           | BERTScore                          | 
| --------- | -------------- | ---------------------------------- | 
| High BLEU | High BERTScore | Safe paraphrasing                  |                
| Low BLEU  | High BERTScore | Excellent paraphrasing             |                
| High BLEU | Low BERTScore  | Copying words but altering meaning |                
| Low BLEU  | Low BERTScore  | Poor paraphrasing                  |                


**BLEU, ROUGE, BERTScore**
---

**BLEU** (Bilingual Evaluation Understudy) and **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) are both automatic metrics for evaluating generated text by comparing it against one or more human-written reference texts, but they emphasize different things and are used for different tasks.

**BLEU** was designed for machine translation. It measures **precision**: of the n-grams (word sequences) the generated text produced, how many also appear in the reference? It checks overlap at multiple n-gram lengths (commonly 1 through 4 words) and combines them, with a penalty if the generated text is too short (to discourage gaming the score by outputting only a few "safe" words). The intuition: a good translation should mostly say things that match what a human translator said.

**ROUGE** was designed for summarization. It measures **recall**: of the n-grams (or longer subsequences) in the reference text, how many did the generated text manage to capture? Common variants include ROUGE-N (n-gram overlap), ROUGE-L (longest common subsequence, capturing word order loosely), and ROUGE-S (skip-bigram overlap). The intuition: a good summary should cover the key content that's actually in the reference, even if phrased a bit differently.

| | BLEU | ROUGE |
|---|---|---|
| Primary use | Machine translation | Summarization |
| Core focus | Precision (generated text matches reference) | Recall (reference content is captured) |
| Typical unit | n-grams (1-4 words), with brevity penalty | n-grams, longest common subsequence, skip-bigrams |

Both share the same core limitation: they rely on exact surface-level word/n-gram matching against reference text, so they penalize valid paraphrases and synonyms that don't share exact wording, even when meaning is preserved. This is one motivation for embedding-based evaluation metrics (e.g., BERTScore), which compare meaning via dense vectors rather than exact word overlap — a direct connection back to the word embedding methods we covered earlier, now applied to *evaluating* generated text rather than representing input text.

**BLEU Score Benchmarks**
| BLEU Score | Interpretation                                |
| ---------- | --------------------------------------------- |
| **90–100** | Almost identical (e.g., trivial rewording)    |
| **70–89**  | Very close; high-quality paraphrase or MT     |
| **50–69**  | Good overlap, some variation (common range)   |
| **30–49**  | Acceptable — more syntactic or lexical shifts |
| **<30**    | Very different — maybe creative, maybe wrong  |

In machine translation or paraphrasing:
BLEU ≥ 60 is often very strong
BLEU ≥ 40 is decent
BLEU < 30 may still be fine for very novel or abstract outputs


**ROUGE Score Benchmarks**
| ROUGE Score (F1) | Interpretation                        |
| ---------------- | ------------------------------------- |
| **0.9 – 1.0**    | Almost exact overlap                  |
| **0.7 – 0.89**   | Strong match — captures most info     |
| **0.5 – 0.69**   | Moderate similarity                   |
| **0.3 – 0.49**   | Partial recall — diverges in content  |
| **<0.3**         | Weak — structure/meaning is different |



**BERTScore**

BERTScore is often confusing because there is no official "excellent/good/bad" range like an exam grade.

The score depends on:

the language
the pretrained model used (roberta-large, deberta, etc.)
the dataset
the task itself

For English paraphrase tasks, people usually interpret the F1 score (the main metric) using empirical ranges.

| BERTScore F1 | Interpretation                    | Quality                          |
| ------------ | --------------------------------- | -------------------------------- |
| ≥ 0.95       | Nearly identical sentences        | Excellent / possibly too similar |
| 0.92 – 0.95  | Very strong semantic preservation | Excellent                        |
| 0.88 – 0.92  | Good paraphrase                   | Good                             |
| 0.84 – 0.88  | Acceptable paraphrase             | Moderate                         |
| 0.80 – 0.84  | Weak semantic preservation        | Poor                             |
| < 0.80       | Meaning likely changed            | Unsatisfactory                   |


Read more about ROUGE, BLEAU and other metrics

https://www.digitalocean.com/community/tutorials/automated-metrics-for-evaluating-generated-text

https://clementbm.github.io/theory/2021/12/23/rouge-bleu-scores.html